# Problem Set 6 - Question 1 (Reconstructed)

This notebook reconstructs the analysis shown in the submitted PS6 Q1 PDF. It studies Monte Carlo convergence for standard-normal draws, simulates terminal index prices under risk-neutral geometric Brownian motion, tests simulated means against their theoretical values, prices a European put by Monte Carlo, and compares an at-the-money trader approximation with the Black-Scholes-Merton benchmark.

The original notebook loads `SPXVols.csv` from the UCLA Derivatives folder. The displayed Q1 calculations do not otherwise depend on the volatility data, but the loading and inspection step is retained here to match the original workflow.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import norm, t as student_t

SEED = 42
rng = np.random.RandomState(SEED)  # reproduces the original np.random.seed output
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

ModuleNotFoundError: No module named 'matplotlib'

## Load the SPX implied-volatility data

The first path matches the location on the original Windows computer. If the file is not found there, the code looks for `SPXVols.csv` in the same folder as the notebook.

In [ ]:
windows_data_path = Path(r"C:\Users\secar\OneDrive\Documents\UCLA\Derivatives\SPXVols.csv")
local_data_path = Path("SPXVols.csv")

if windows_data_path.exists():
    spx_vols_path = windows_data_path
elif local_data_path.exists():
    spx_vols_path = local_data_path
else:
    raise FileNotFoundError(
        "SPXVols.csv was not found. Save it in "
        r"C:\Users\secar\OneDrive\Documents\UCLA\Derivatives "
        "or in the same folder as this notebook."
    )

spx_vols = pd.read_csv(
    spx_vols_path,
    header=None,
    names=["Strike", "ImpliedVol"],
)

print(f"Loaded {len(spx_vols):,} observations from: {spx_vols_path}")
print("\nFirst five rows:")
print(spx_vols.head())
print("\nLast five rows:")
print(spx_vols.tail())
print("Shape:", spx_vols.shape)

## 1. Convergence of standard-normal sample statistics

Generate one million standard-normal draws and use nested samples so each larger sample contains every observation in the smaller samples.

In [ ]:
sample_sizes = [1_000, 10_000, 100_000, 1_000_000]
z_full = rng.normal(0, 1, max(sample_sizes))
z_sequences = {n: z_full[:n] for n in sample_sizes}

def normal_sample_stats(z):
    return {
        "Sample Mean": np.mean(z),
        "Sample Variance": np.var(z, ddof=1),
        "Sample Std Dev": np.std(z, ddof=1),
    }

rows = []
for n, z in z_sequences.items():
    row = {"n": n}
    row.update(normal_sample_stats(z))
    rows.append(row)

normal_stats_df = pd.DataFrame(rows)
normal_stats_df["True Mean"] = 0.0
normal_stats_df["True Variance"] = 1.0
normal_stats_df["True Std Dev"] = 1.0
normal_stats_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(normal_stats_df["n"], normal_stats_df["Sample Mean"], marker="o", label="Sample mean")
ax.axhline(0, linestyle="--", color="black", label="True mean")
ax.set_xscale("log")
ax.set_xlabel("Sample size n")
ax.set_ylabel("Mean")
ax.set_title("Convergence of Sample Mean for N(0,1) Draws")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

The sample statistics move toward the population values $(0,1,1)$ as $n$ increases, consistent with the Law of Large Numbers.

## 2. Simulating the terminal index price

Under the risk-neutral measure,

$$S_T=S_0\exp\left[(r-y-\tfrac12\sigma^2)T+\sigma\sqrt{T}Z\right],\qquad Z\sim N(0,1).$$

Here, $y$ is the continuous dividend yield.

In [ ]:
S0 = 6_835.0
K = 6_835.0
T = 1.0
sigma = 0.19
r = 0.0375
y = 0.0095

mu_q = r - y - 0.5 * sigma**2
true_ST_mean = S0 * np.exp((r - y) * T)
true_log_mean = np.log(S0) + mu_q * T
true_log_var = sigma**2 * T
true_ST_var = S0**2 * np.exp(2 * (r - y) * T) * (np.exp(sigma**2 * T) - 1)
true_ST_std = np.sqrt(true_ST_var)

pd.Series({
    "E[S_T]": true_ST_mean,
    "Std[S_T]": true_ST_std,
    "E[log(S_T)]": true_log_mean,
    "Var[log(S_T)]": true_log_var,
})

In [ ]:
def simulate_ST(z):
    return S0 * np.exp(mu_q * T + sigma * np.sqrt(T) * z)

ST_sequences = {n: simulate_ST(z) for n, z in z_sequences.items()}

def terminal_price_stats(ST):
    log_ST = np.log(ST)
    return {
        "Sample Mean(ST)": np.mean(ST),
        "Sample Std(ST)": np.std(ST, ddof=1),
        "Sample Log Mean": np.mean(log_ST),
        "Sample Log Variance": np.var(log_ST, ddof=1),
    }

rows = []
for n, ST in ST_sequences.items():
    row = {"n": n}
    row.update(terminal_price_stats(ST))
    rows.append(row)

ST_stats_df = pd.DataFrame(rows)
ST_stats_df["True Mean(ST)"] = true_ST_mean
ST_stats_df["True Std(ST)"] = true_ST_std
ST_stats_df["True Log Mean"] = true_log_mean
ST_stats_df["True Log Variance"] = true_log_var
ST_stats_df

The simulated log-prices approach their theoretical normal moments, while simulated terminal prices approach the corresponding lognormal moments.

## 3. One-sample t-tests

For each sample size, test whether the simulated mean terminal price and mean log-price differ significantly from their theoretical values.

In [ ]:
def one_sample_t_test(sample, hypothesized_mean):
    n = len(sample)
    sample_mean = np.mean(sample)
    sample_std = np.std(sample, ddof=1)
    t_stat = (sample_mean - hypothesized_mean) / (sample_std / np.sqrt(n))
    p_value = 2 * student_t.sf(abs(t_stat), df=n - 1)
    return t_stat, p_value

rows = []
for n, ST in ST_sequences.items():
    t_ST, p_ST = one_sample_t_test(ST, true_ST_mean)
    t_log, p_log = one_sample_t_test(np.log(ST), true_log_mean)
    rows.append({
        "n": n,
        "t-stat for E[ST]": t_ST,
        "p-value for E[ST]": p_ST,
        "t-stat for E[log ST]": t_log,
        "p-value for E[log ST]": p_log,
    })

test_results_df = pd.DataFrame(rows)
test_results_df

In [ ]:
alpha = 0.05
p_columns = ["p-value for E[ST]", "p-value for E[log ST]"]
if (test_results_df[p_columns] > alpha).all().all():
    print("At the 5% level, we fail to reject every null hypothesis.")
else:
    print("At least one null hypothesis is rejected at the 5% level.")

## 4. Black-Scholes-Merton and Monte Carlo European put values

In [ ]:
def bsm_put_price(S0, K, T, r, y, sigma):
    d1 = (np.log(S0 / K) + (r - y + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S0 * np.exp(-y * T) * norm.cdf(-d1)

put_bsm = bsm_put_price(S0, K, T, r, y, sigma)
print(f"BSM European put value: {put_bsm:,.6f}")

In [ ]:
def mc_put_price_and_se(ST):
    payoff = np.maximum(K - ST, 0.0)
    discounted_payoff = np.exp(-r * T) * payoff
    price = np.mean(discounted_payoff)
    se = np.std(discounted_payoff, ddof=1) / np.sqrt(len(discounted_payoff))
    return price, se

rows = []
for n, ST in ST_sequences.items():
    price, se = mc_put_price_and_se(ST)
    rows.append({
        "n": n,
        "MC Put Price": price,
        "CLT Std Error": se,
        "95% CI Lower": price - 1.96 * se,
        "95% CI Upper": price + 1.96 * se,
        "Error vs BSM": price - put_bsm,
    })

mc_put_df = pd.DataFrame(rows)
mc_put_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].errorbar(
    mc_put_df["n"], mc_put_df["MC Put Price"],
    yerr=1.96 * mc_put_df["CLT Std Error"], fmt="o-", capsize=4,
    label="Monte Carlo 95% CI"
)
axes[0].axhline(put_bsm, color="black", linestyle="--", label="BSM benchmark")
axes[0].set_xscale("log")
axes[0].set_xlabel("Sample size n")
axes[0].set_ylabel("European put value")
axes[0].set_title("Monte Carlo Price Convergence")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].loglog(mc_put_df["n"], mc_put_df["CLT Std Error"], marker="o", label="Observed SE")
reference = mc_put_df["CLT Std Error"].iloc[0] * np.sqrt(mc_put_df["n"].iloc[0] / mc_put_df["n"])
axes[1].loglog(mc_put_df["n"], reference, linestyle="--", label=r"$n^{-1/2}$ reference")
axes[1].set_xlabel("Sample size n")
axes[1].set_ylabel("Monte Carlo standard error")
axes[1].set_title("Standard-Error Convergence")
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

The Monte Carlo estimate approaches the analytical BSM value as the sample grows, while its standard error decreases at approximately the expected $n^{-1/2}$ rate.

## 5. At-the-money trader approximation

The approximation used in the original work first estimates the at-the-money call value and then applies put-call parity.

In [ ]:
call_trader = S0 * np.exp(-y * T) * sigma * np.sqrt(T / (2 * np.pi))
put_trader = call_trader - S0 * np.exp(-y * T) + K * np.exp(-r * T)
relative_error = abs(put_trader - put_bsm) / put_bsm

trader_comparison_df = pd.DataFrame({
    "Quantity": ["BSM Put", "Trader Approx Put", "Relative Error"],
    "Value": [put_bsm, put_trader, relative_error],
})
trader_comparison_df

For these inputs, the trader approximation materially understates the exact BSM put value, illustrating the cost of using a simple shortcut instead of the full option-pricing formula.

## Optional FX interpretation

The same model can be applied to an FX option by interpreting $S_0$ as a spot exchange rate, $r$ as the domestic interest rate, and $y$ as the foreign interest rate. Under that mapping, the BSM expression is the Garman-Kohlhagen currency-option model.